# ==========================================================
# AI-POWERED CUSTOMER RETENTION INTELLIGENCE PLATFORM
#
# PHASE 8
# RETENTION RECOMMENDATION ENGINE
#
# NOTEBOOK:
# 01_Recommendation_Setup.ipynb
# ==========================================================

In [1]:
# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import os
import joblib
import numpy as np
import pandas as pd

print("=" * 60)
print("LIBRARIES IMPORTED SUCCESSFULLY")
print("=" * 60)

LIBRARIES IMPORTED SUCCESSFULLY


In [2]:
# ==========================================================
# PROJECT PATH
# ==========================================================

PROJECT_ROOT = os.path.abspath("..")

print(PROJECT_ROOT)

d:\AI-Powered-Customer-Retention-Intelligence-Platform


In [3]:
# ==========================================================
# LOAD MODEL DATASET
# ==========================================================

dataset_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "customer_retention_model_data.csv"
)

customer_data = pd.read_csv(dataset_path)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print(customer_data.head())

print("\nShape")
print(customer_data.shape)

DATASET LOADED
                 customer_unique_id  Recency  Frequency  Monetary  \
0  0000366f3b9a7992bf8c76cfdf3221e2      161          1    141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f      164          1     27.19   
2  0000f46a3911fa3c0805444483337064      586          1     86.22   
3  0000f6ccb0745a6a4b88665a16c9f078      370          1     43.62   
4  0004aac84e0df4da2b147fca70cf8255      337          1    196.89   

   average_review_score preferred_payment_method  average_order_value  \
0                   5.0              credit_card               141.90   
1                   4.0              credit_card                27.19   
2                   3.0              credit_card                86.22   
3                   4.0              credit_card                43.62   
4                   5.0              credit_card               196.89   

   customer_lifetime_value Customer_Risk  churn_label  
0                   141.90        Medium            0  
1                  

In [12]:
# ==========================================================
# LOAD CHAMPION MODEL
# ==========================================================

model_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "gradient_boosting.pkl"   # ✅ Correct filename
)

gb_model = joblib.load(model_path)

print("=" * 60)
print("GRADIENT BOOSTING MODEL LOADED")
print("=" * 60)

GRADIENT BOOSTING MODEL LOADED


In [10]:
import os

print(os.path.exists(r"D:\models\gradient_boosting.pkl"))
print(os.path.exists(r"D:\AI-Powered-Customer-Retention-Intelligence-Platform\models\gradient_boosting.pkl"))

False
True


In [13]:
# ==========================================================
# PREPARE FEATURES
# ==========================================================

X = customer_data.drop(
    columns=["churn_label"]
)

print(X.head())

                 customer_unique_id  Recency  Frequency  Monetary  \
0  0000366f3b9a7992bf8c76cfdf3221e2      161          1    141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f      164          1     27.19   
2  0000f46a3911fa3c0805444483337064      586          1     86.22   
3  0000f6ccb0745a6a4b88665a16c9f078      370          1     43.62   
4  0004aac84e0df4da2b147fca70cf8255      337          1    196.89   

   average_review_score preferred_payment_method  average_order_value  \
0                   5.0              credit_card               141.90   
1                   4.0              credit_card                27.19   
2                   3.0              credit_card                86.22   
3                   4.0              credit_card                43.62   
4                   5.0              credit_card               196.89   

   customer_lifetime_value Customer_Risk  
0                   141.90        Medium  
1                    27.19        Medium  
2                

In [14]:
# ==========================================================
# PREDICT RISK PROBABILITY
# ==========================================================

risk_probability = gb_model.predict_proba(X)[:, 1]

customer_data["Risk_Probability"] = risk_probability

print("=" * 60)
print("RISK PROBABILITIES GENERATED")
print("=" * 60)

print(customer_data[
    ["Risk_Probability"]
].head())

RISK PROBABILITIES GENERATED
   Risk_Probability
0          0.683347
1          0.617935
2          0.777345
3          0.748424
4          0.677529


In [15]:
# ==========================================================
# CREATE RISK LEVELS
# ==========================================================

def risk_level(probability):

    if probability >= 0.70:
        return "High"

    elif probability >= 0.40:
        return "Medium"

    else:
        return "Low"


customer_data["Risk_Level"] = (
    customer_data["Risk_Probability"]
    .apply(risk_level)
)

print("=" * 60)
print("RISK LEVEL DISTRIBUTION")
print("=" * 60)

print(
    customer_data["Risk_Level"]
    .value_counts()
)

RISK LEVEL DISTRIBUTION
Risk_Level
High      58757
Medium    37162
Low         177
Name: count, dtype: int64


In [16]:
# ==========================================================
# BUSINESS PRIORITY
# ==========================================================

priority_map = {

    "High": "Critical",

    "Medium": "Medium",

    "Low": "Low"

}

customer_data["Priority"] = (
    customer_data["Risk_Level"]
    .map(priority_map)
)

print(customer_data[
    [
        "Risk_Probability",
        "Risk_Level",
        "Priority"
    ]
].head())

   Risk_Probability Risk_Level  Priority
0          0.683347     Medium    Medium
1          0.617935     Medium    Medium
2          0.777345       High  Critical
3          0.748424       High  Critical
4          0.677529     Medium    Medium


In [17]:
# ==========================================================
# RECOMMENDATION DATASET
# ==========================================================

recommendation_dataset = customer_data.copy()

print("=" * 60)
print("RECOMMENDATION DATASET READY")
print("=" * 60)

print(recommendation_dataset.head())

print("\nShape")
print(recommendation_dataset.shape)

RECOMMENDATION DATASET READY
                 customer_unique_id  Recency  Frequency  Monetary  \
0  0000366f3b9a7992bf8c76cfdf3221e2      161          1    141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f      164          1     27.19   
2  0000f46a3911fa3c0805444483337064      586          1     86.22   
3  0000f6ccb0745a6a4b88665a16c9f078      370          1     43.62   
4  0004aac84e0df4da2b147fca70cf8255      337          1    196.89   

   average_review_score preferred_payment_method  average_order_value  \
0                   5.0              credit_card               141.90   
1                   4.0              credit_card                27.19   
2                   3.0              credit_card                86.22   
3                   4.0              credit_card                43.62   
4                   5.0              credit_card               196.89   

   customer_lifetime_value Customer_Risk  churn_label  Risk_Probability  \
0                   141.90        Medium  

In [18]:
# ==========================================================
# SAVE DATASET
# ==========================================================

output_path = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "customer_recommendation_data.csv"
)

recommendation_dataset.to_csv(
    output_path,
    index=False
)

print("=" * 60)
print("DATASET SAVED")
print("=" * 60)

print(output_path)

DATASET SAVED
d:\AI-Powered-Customer-Retention-Intelligence-Platform\data\processed\customer_recommendation_data.csv
